In [ ]:
### Imports

import pathlib
import pickle

from usv_playpen.modeling.modeling_usv_manifold_position import ContinuousModelingPipeline, DeepContinuousModelRunner
from usv_playpen.modeling.modeling_vocal_categories_multinomial import MultinomialModelingPipeline
from usv_playpen.modeling.modeling_vocal_bout_parameters import BoutParameterPipeline
from usv_playpen.modeling.modeling_vocal_onsets import GeneralizedLinearModelPipeline
from usv_playpen.modeling.model_selection import bout_onset_model_selection, vocal_category_model_selection
from usv_playpen.modeling.modeling_vocal_categories import VocalCategoryModelingPipeline
from usv_playpen.modeling.modeling_plots import (plot_feature_ranking, plot_significant_filters, plot_significant_filters_grid,
                                                     plot_raw_feature_difference, plot_model_selection_results,
                                                     plot_univariate_multinomial_performance, plot_multinomial_selection_trajectory,
                                                     plot_multinomial_multivariate_filters, plot_multinomial_selection_diagnosis,
                                                     DeepResultsVisualizer, plot_vocalization_embedding_space)


In [ ]:
### Extract and save vocal bout onset to model

GeneralizedLinearModelPipeline(modeling_settings_dict=None).extract_and_save_modeling_input_data_()

In [ ]:
### Extract and save category prediction data to model (NB: remove target_category argument to extract for all categories!)

VocalCategoryModelingPipeline(modeling_settings_dict=None).extract_and_save_category_input_data(target_category=18)

In [ ]:
### Extract and save category prediction data for multinomial modeling

MultinomialModelingPipeline(modeling_settings_dict=None).extract_and_save_multinomial_input_data()

In [ ]:
### Extract and save vocal bout duration/complexity data to model

BoutParameterPipeline(modeling_settings_dict=None).extract_and_save_modeling_input_data_()

In [ ]:
### Extract and save USV UMAP target position data to model

ContinuousModelingPipeline(modeling_settings_dict=None).extract_and_save_continuous_data()

In [ ]:
### Perform model selection for bout onset prediction

bout_onset_model_selection(univariate_results_path='/mnt/falkner/Bartul/modeling/gam_results_female_bout_lam=null_max_iter=500_n_splines_time=8_n_splines_value=5.pkl',
                           input_data_path='/mnt/falkner/Bartul/modeling/glm_pdata_female_20251203_161303_110sess_bout_hist4s.pkl',
                           output_directory='/mnt/falkner/Bartul/modeling/model_selection_results/bout_onset/male/intact_partner',
                           use_top_rank_as_anchor=True,
                           p_val=0.01)

In [ ]:
### Perform model selection for USV categories

vocal_category_model_selection(univariate_results_path='/mnt/falkner/Bartul/modeling/gam_results_male_category_3_lam=0.6_max_iter=500_n_splines_time=8_n_splines_value=5.pkl',
                               input_data_path='/mnt/falkner/Bartul/modeling/modeling_category_3_male_FullSyntax_20260109_120031_hist4s.pkl',
                               output_directory='/mnt/falkner/Bartul/modeling/model_selection_results/vocal_categories/male/intact_partner',
                               use_top_rank_as_anchor=True,
                               p_val=0.01)

In [ ]:
### Save individual results files into one big file for easier manipulation

results_files = pathlib.Path('/mnt/scratch/Bartul/modeling/univariate_results/multinomial_category/male_mute_partner').glob('*.pkl')

results_files = list(results_files)
results_files.sort(key=lambda f: int(f.name.split('_')[1]))

new_save_dict = {}
for results_file in results_files:
    with open(results_file, 'rb') as pickle_file:
        modeling_data = pickle.load(pickle_file)

    for feat in modeling_data.keys():
        new_save_dict[feat] = modeling_data[feat]

with open('/mnt/scratch/Bartul/modeling/univariate_results/multinomial_category/univariate_multinomial_category_male_mute_partner_bin_resize=10_lambda=1E0_l2=1E-1.pkl', 'wb') as output_file:
    pickle.dump(new_save_dict, output_file)

In [ ]:
### Plot feature ranking for univariate models (each metric gets its own plot)

plot_feature_ranking(results_file_loc='/mnt/falkner/Bartul/modeling/univariate_results/category_3_gam_results_female_splits=100_lam=0.6_max_iter=100_n_splines_time=8_n_splines_value=5.pkl',
                     p_val=0.01,
                     evaluation_metric='ll',
                     evaluation_metric_name='Negative Log-Likelihood',
                     secondary_metric='score',
                     secondary_metric_name="Accuracy",
                     ignore_features=None,
                     save_plot=False,
                     output_dir='/mnt/falkner/Bartul/modeling/figures')


In [ ]:
### Plot individual filters of univariate models

plot_significant_filters(results_file_loc='/mnt/falkner/Bartul/modeling/gam_results_male_mute_partner_category_18_lam=0.6_max_iter=500_n_splines_time=8_n_splines_value=5.pkl',
                         metric='ll',
                         ignore_features=None,
                         p_val=0.01,
                         save_plot=False,
                         output_dir='/mnt/falkner/Bartul/modeling/figures')

In [ ]:
### Plot all significant filters in a grid, baseline-corrected, considering univariate models

plot_significant_filters_grid(results_file_loc='/mnt/falkner/Bartul/modeling/gam_results_male_category_10_lam=0.6_max_iter=500_n_splines_time=8_n_splines_value=5.pkl',
                              ignore_features=None,
                              metric='ll',
                              p_val_threshold=0.01,
                              save_plot=False,
                              output_dir='/mnt/falkner/Bartul/modeling/figures')

In [ ]:
### Plot raw (z-scored) feature differences

plot_raw_feature_difference(pickle_file_path='/mnt/falkner/Bartul/modeling/modeling_category_3_male_presence_all_20260129_193701_hist4s.pkl',
                            feature_key='self.neck_elevation',
                            feature_color='#9AC0CD',
                            subset_fraction=0.05,
                            n_bootstraps=100,
                            save_plots=False,
                            output_dir='/mnt/falkner/Bartul/modeling/figures')

In [ ]:
### Plot model selection results

plot_model_selection_results(selection_results_dir='/mnt/falkner/Bartul/modeling/model_selection_results/bout_params/bout_onset/female',
                             metric_secondary='auc',
                             save_plots=False,
                             output_dir='/mnt/falkner/Bartul/modeling/figures')



In [ ]:
### Plot performance of univariate multinomial models

plot_univariate_multinomial_performance(
        results_file_loc='/mnt/scratch/Bartul/modeling/univariate_results/multinomial_categories/univariate_multinomial_categories_male_bin_resize=10_lambda=1E0_l2=1E-1.pkl',
        evaluation_metric='auc',
        evaluation_metric_name='Area Under the ROC Curve',
        secondary_metric='score',
        secondary_metric_name='Balanced Accuracy',
        p_val_threshold=0.05,
        base_cmap='inferno',
        diff_cmap='bwr',
        save_plot=False,
        output_dir='/mnt/falkner/Bartul/modeling/figures')

In [ ]:
### Plot performance trajectory of model selection for multinomial models

plot_multinomial_selection_trajectory(
        selection_results_dir='/mnt/scratch/Bartul/modeling/model_selection_results/multinomial_category/male_mute_partner',
        metric_primary='auc',
        primary_metric_name='Area Under the ROC Curve',
        metric_secondary='score',
        secondary_metric_name="Balanced Accuracy",
        save_plot=False,
        output_dir='/mnt/falkner/Bartul/modeling/figures')

In [ ]:
### Plot filters of multinomial models (final model selection)

plot_multinomial_multivariate_filters(
        selection_results_dir='/mnt/scratch/Bartul/modeling/model_selection_results/multinomial_category/male',
        history_window_sec=4.0,
        cmap='bwr',
        save_plot=True,
        output_dir='/mnt/falkner/Bartul/modeling/figures'
)

In [ ]:
### Plot how much te final model selection for multinomial models differs from the top-ranked univariate model

plot_multinomial_selection_diagnosis(
        selection_results_dir='/mnt/scratch/Bartul/modeling/model_selection_results/multinomial_category/male_mute_partner',
        cmap_base='inferno',
        cmap_diff='bwr',
        save_plot=True,
        output_dir='/mnt/falkner/Bartul/modeling/figures'
)

In [ ]:
import pathlib
import pickle
import glob
import re
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from matplotlib.colors import SymLogNorm

def plot_multinomial_log_confusion_matrix(
        selection_results_dir: str,
        cmap: str = 'inferno',
        save_plot: bool = False,
        output_dir: str = None
) -> None:

    # 1. FORCE White Background / Black Text
    TEXT_COLOR = '#000000'
    BG_COLOR = '#FFFFFF'

    plt.rcParams.update({
        'text.color': TEXT_COLOR,
        'axes.labelcolor': TEXT_COLOR,
        'xtick.color': TEXT_COLOR,
        'ytick.color': TEXT_COLOR,
        'axes.edgecolor': TEXT_COLOR,
        'figure.facecolor': BG_COLOR,
        'axes.facecolor': BG_COLOR
    })

    file_pattern = pathlib.Path(selection_results_dir) / "*_step_*.pkl"
    files = sorted(glob.glob(str(file_pattern)),
                   key=lambda x: int(re.search(r'_step_(\d+)', x).group(1)))

    if not files:
        print(f"No step files found in {selection_results_dir}")
        return

    # 2. Extract Final Model Predictions
    valid_data = None
    for fpath in reversed(files):
        with open(fpath, 'rb') as f:
            data = pickle.load(f)
        if data.get('selected_feature') is not None:
            valid_data = data
            break

    if valid_data is None:
        print("No valid features were selected in this run.")
        return

    winner = valid_data['selected_feature']
    winner_data = valid_data['candidates_summary'][winner]
    folds_data = winner_data['folds']

    # --- ROBUST CLASS EXTRACTION ---
    if 'classes' in valid_data:
        classes = valid_data['classes']
    elif 'classes' in winner_data:
        classes = winner_data['classes']
    else:
        # Absolute fallback: mathematically infer from the unique true labels
        print("Explicit 'classes' list not found. Inferring from data...")
        unique_vals = np.unique(np.concatenate(folds_data['y_true']))
        classes = [str(int(v)) for v in unique_vals]

    # Concatenate all cross-validation folds to get the global absolute count
    y_true = np.concatenate(folds_data['y_true'])
    y_pred = np.concatenate(folds_data['y_pred'])

    # 3. Calculate Raw Confusion Matrix (No Normalization)
    cm_raw = confusion_matrix(y_true, y_pred)

    # 4. Plotting
    fig, ax = plt.subplots(figsize=(8, 6.5), dpi=300)
    fig.patch.set_facecolor(BG_COLOR)
    ax.set_facecolor(BG_COLOR)

    # SymLogNorm: linthresh=1 means 0 to 1 is linear (safe for zeros), >1 is logarithmic
    log_norm = SymLogNorm(linthresh=1, vmin=0, vmax=cm_raw.max())

    # fmt="d" forces integers
    sns.heatmap(cm_raw, annot=True, fmt="d", cmap=cmap, ax=ax,
                xticklabels=classes, yticklabels=classes,
                norm=log_norm,
                cbar_kws={'label': 'Absolute Count (Log10 Scale)', 'shrink': 0.8},
                annot_kws={"size": 11, "weight": "bold"})

    # 5. Formatting & Labels
    # Safely get feature count
    n_feats = len(valid_data.get('final_model_features', valid_data.get('current_features', []) + [winner]))
    ax.set_title(f"Final Multivariate Model ({n_feats} Features)\nRaw Absolute Error Distribution",
                 fontsize=14, fontweight='bold', color=TEXT_COLOR, pad=20)

    ax.set_xlabel('Predicted USV Category', fontsize=12, fontweight='bold', labelpad=15)
    ax.set_ylabel('True USV Category', fontsize=12, fontweight='bold', labelpad=15)

    # Draw strict black borders and explicitly kill minor ticks
    for _, spine in ax.spines.items():
        spine.set_visible(True)
        spine.set_color(TEXT_COLOR)
        spine.set_linewidth(1.5)

    ax.tick_params(which='major', colors=TEXT_COLOR, labelsize=10, bottom=True, left=True)
    ax.tick_params(which='minor', bottom=False, left=False, top=False, right=False) # Kills the weird boundary ticks

    # Ensure Colorbar matches styling
    cbar = ax.collections[0].colorbar
    cbar.ax.yaxis.set_tick_params(color=TEXT_COLOR, labelcolor=TEXT_COLOR)
    cbar.set_label('Absolute Count (Log Scale)', color=TEXT_COLOR, fontsize=11, fontweight='bold', labelpad=15)

    plt.tight_layout()

    # 6. Save Logic
    if save_plot:
        out_dir = pathlib.Path(output_dir) if output_dir else pathlib.Path(selection_results_dir)
        out_dir.mkdir(parents=True, exist_ok=True)

        path_str = str(selection_results_dir).lower()
        condition = 'male_mute_partner' if 'male_mute_partner' in path_str else \
                    ('female' if 'female' in path_str else 'male')

        fname = f"model_selection_multinomial_usv_category_{condition}_raw_log_cm.svg"
        fig.savefig(out_dir / fname, facecolor=BG_COLOR, bbox_inches='tight')
        print(f"Log-scaled raw confusion matrix saved to: {fname}")

    plt.show()

In [ ]:
plot_multinomial_log_confusion_matrix(
        selection_results_dir='/mnt/scratch/Bartul/modeling/model_selection_results/multinomial_category/male',
        cmap='inferno',
        save_plot=False,
        output_dir=None
)

In [ ]:
### Run CNN to predict vocal manifold positions

from jax_neural_network_cnn import NeuralContinuousCNNRunner
runner = NeuralContinuousCNNRunner(modeling_settings=None)

data_blocks = runner.load_multivariate_data_blocks(pkl_path='/mnt/falkner/Bartul/modeling/modeling_UMAP_manifold_position_male_20260226_145551_hist4s.pkl')


In [ ]:
runner.run_cnn_training(data_blocks=data_blocks)

In [ ]:
### Plot CNN results in predicting vocal manifold positions
from usv_playpen.modeling.modeling_plots import DeepResultsVisualizer

deep_visualizer = DeepResultsVisualizer(results_pkl_path='/mnt/falkner/Bartul/modeling/neural_network_results/cnn_manifold_integrated_predictions_male_20260326_220132.pkl',
                                        modeling_settings=None,
                                        visualization_settings=None)

choose_analysis = 'regional_saliency'  # 'error_landscape' 'spatial_precision_grid' 'feature_importance' 'permutation_test'

if choose_analysis == 'permutation_test':
    deep_visualizer.plot_permutation_test(save_plot=False,
                                          output_dir='/mnt/falkner/Bartul/modeling/figures',
                                          file_format='png')
elif choose_analysis == 'feature_importance':
    deep_visualizer.plot_feature_importance(snr_threshold=3.0,
                                            cmap='inferno',
                                            error_bar_color='#000000',
                                            save_plot=False,
                                            output_dir='/mnt/falkner/Bartul/modeling/figures',
                                            file_format='png')
elif choose_analysis == 'spatial_precision_grid':
    deep_visualizer.plot_spatial_precision_grid(n_patches=20,
                                                min_samples=25,
                                                plot_type='density',
                                                bg_pt_color='#E0E0E0',
                                                peak_pt_color='cyan',
                                                square_edge_color='#000000',
                                                panel_fontsize=9,
                                                figsize_unit=3.0,
                                                save_plot=False,
                                                output_dir='/mnt/falkner/Bartul/modeling/figures',
                                                file_format='png')
elif choose_analysis == 'error_landscape':
    deep_visualizer.plot_error_landscape(gridsize=30,
                                         cmap='inferno',
                                         vmax_percentile=95.0,
                                         save_plot=False,
                                         output_dir='/mnt/falkner/Bartul/modeling/figures',
                                         file_format='png')
elif choose_analysis == 'regional_saliency':
    deep_visualizer.plot_regional_saliency_inset(
        source_data_path='',
        region_key='categories_1_and_2',
        category_name='Complex USVs',
        prediction_plot_type='density',
        save_plot=False,
        output_dir='/mnt/falkner/Bartul/modeling/figures',
        file_format='png')
else:
    print(f"Option {choose_analysis} not recognized.")




In [ ]:

plot_vocalization_embedding_space(umap_position_file_path='/mnt/falkner/Charlie/USV_town/h5_files_ch/whole_dset_plus_ephys/session_to_umap_dict.h5',
                                  cluster_category_file_path='/mnt/falkner/Charlie/USV_town/h5_files_ch/whole_dset_plus_ephys/clustering/cluster_dict_6clus.h5',
                                  target_subdirs=["Liza/data", "Jinrun/Data", "Bartul/Data"],
                                  csv_category_column_id='usv_supercategory',
                                  spec_cmap_name='inferno',
                                  x_range=(15.0, 17.5),
                                  y_range=(4.5, 6.5),
                                  grid_dims=(4, 4),
                                  save_plot=False,
                                  output_dir='/mnt/falkner/Bartul/modeling/figures')


In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

def plot_binary_classification_accuracy(final_step_path: str):
    """
    Plots a normalized confusion matrix showing the percentage of
    Simple vs. Complex USVs correctly classified by the Stage 1 model.
    """

    # 1. Load the final promoted step data
    with open(final_step_path, 'rb') as f:
        data = pickle.load(f)

    winner = data['selected_feature']
    if winner is None:
        winner = data['current_features'][-1]

    # 2. Aggregate y_true and y_pred across all CV folds
    # We use a 0.5 threshold as defined in your selection logic
    all_y_true = []
    all_y_pred = []

    fold_data = data['candidates_summary'][winner]['folds']

    for i in range(len(fold_data['y_true'])):
        y_true = fold_data['y_true'][i]
        y_probs = fold_data['y_probs'][i]

        # Binary prediction based on the 0.5 threshold
        y_pred = (y_probs >= 0.5).astype(int)

        all_y_true.extend(y_true)
        all_y_pred.extend(y_pred)

    all_y_true = np.array(all_y_true)
    all_y_pred = np.array(all_y_pred)

    # 3. Calculate Confusion Matrix
    # Labels: 0 = Simple (3,4,5), 1 = Complex (1,2,6)
    cm = confusion_matrix(all_y_true, all_y_pred)

    # Normalize by the number of USVs in each true class to get percentages
    cm_perc = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100

    # 4. Visualization
    plt.figure(figsize=(6, 5), dpi=120)
    sns.heatmap(
        cm_perc,
        annot=True,
        fmt=".1f",
        cmap="Blues",
        cbar_kws={'label': 'Classification Accuracy (%)'},
        xticklabels=['Simple', 'Complex'],
        yticklabels=['Simple', 'Complex']
    )

    plt.title(f"Stage 1 Gatekeeper Performance\nFeature Set: {len(data['current_features'])} Winning Predictors", pad=15)
    plt.xlabel("Predicted Family")
    plt.ylabel("True Family")
    plt.tight_layout()

    plt.show()

# Run the plot (replace with your actual successful Step 10 path)
plot_binary_classification_accuracy("/mnt/scratch/Bartul/modeling/model_selection_results/multinomial_category/male_mute_partner/model_selection_hierarchical_stage1_mixed_step_10.pkl")